In [1]:
# Imports section
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
df = pd.read_csv("science_data_large.csv")

print(df.head(15))
print(df.describe())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   226.750000  1.298267e+05
50%        500.500

## Part 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)

X = df[["Temperature °C", "Mols KCL"]]
y = df["Size nm^3"]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=69)


## Part 3. Perform a Linear Regression

In [4]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
X_dummy = [500, 471.53]
print("data point", X_dummy, "prediction:", model.predict([X_dummy]))

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print("score:",model.score(X_test, y_test))

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print("coefficients", model.coef_)
print("bias", model.intercept_)

data point [500, 471.53] prediction: [510983.41022426]
score: 0.835431726595059
coefficients [ 873.24723627 1031.92974219]
bias -412226.0392449613


c:\Python311\Lib\site-packages\sklearn\base.py:409: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


The score on the test set is 0.83543, which indicates the coefficient of determination $R^2$. This means that 83.5% of variance in the slime size can be explained by a linear model with temperature and KCL. This is a pretty good score, since the linear model is capturing most of the patterns in the data, but it could still be improved.

Hypothesis: $h(x) = -412226.03924 + 873.24723 x_1 + 1031.92974 x_2$

## Part 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data

cross_val_score(model, X, y, cv=10)

# Report on their finding and their significance

array([0.81123596, 0.86440978, 0.87808742, 0.86561069, 0.87495621,
       0.84484397, 0.87941022, 0.86349411, 0.78353682, 0.88686516])

The `cross_val_score` function uses k-fold validation. In this technique, the data is split into k=10 partitions of 10%. The model is then trained on 9 of these partitions before being tested and scored on the last partition (it's trained on 90% of the data and tested on 10%, just like the original test). This process is repeated so that every partition is given the chance to be the testing partition, and the function returns all 5 of these scores. This method gauges an unbiased sense of how good the model is.

Most of the partitions score in the 80s range, so the model seems to be performing decently under different train and test sets. This is a sign of low model variance and high model stability, which are good things.

## Part 5. Using Polynomial Regression

In [6]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2

poly = PolynomialFeatures(2)
X_poly = poly.fit_transform(X)
X_train_poly = poly.transform(X_train)
X_test_poly = poly.transform(X_test)
#X_train_poly features: 1, x1, x2, x1^2, x1x2, x2^2


model2 = LinearRegression()
model2.fit(X_train_poly, y_train)

# Report on the metrics and output the resultant equation as you did in Part 3.

# Create a sample datapoint and predict the output of that sample with the trained model
print("data point", X_dummy, "prediction:", model2.predict(poly.transform([X_dummy])))

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print("score:",model2.score(X_test_poly, y_test))

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print("coefficients", model2.coef_)
print("bias", model2.intercept_)


data point [500, 471.53] prediction: [483882.58687196]
score: 1.0
coefficients [ 0.00000000e+00  1.20000000e+01 -1.23861031e-07 -4.55542028e-12
  2.00000000e+00  2.85714287e-02]
bias 1.591298496350646e-05


c:\Python311\Lib\site-packages\sklearn\base.py:409: UserWarning: X does not have valid feature names, but PolynomialFeatures was fitted with feature names
  warnings.warn(


In [7]:
#This is the version with cross validation

cross_val_score(model2, X_poly, y)

array([1., 1., 1., 1., 1.])

The score on the test set is 1.0 for both regular testing and k-fold validation, which indicates that the data can be perfectly modeled using polynomial regression.

Coefficients:

$1: 0.00000000e+00\\ x_1: 1.20000000e+01\\ x_2: -1.23861031e-07 \approx 0\\ x_1^2: -4.55542028e-12 \approx 0\\
  x_1x_2: 2.00000000e+00\\  x_2^2: 2.85714287e-02$

Bias: $1.591298496350646e-05 \approx 0$

Hypothesis: $h(x) = 12 x_1 + 2x_1x_2 + 2.85714*10^{-2} x_2^2$